In [2]:
pip install sentence-transformers faiss-cpu pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# Load Dataset


with open("search_text_pipeline.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# لو المنتجات داخل key اسمها products
products = data["products"]

print(f"Total products: {len(products)}")


# Load Embedding Model


model = SentenceTransformer('all-MiniLM-L6-v2')


# Generate Embeddings


embedding_records = []

for idx, product in enumerate(products):

    search_text = product["search_text"]

    
    embedding = model.encode(
        search_text,
        normalize_embeddings=True
    )

    embedding_records.append({
        "product_id": product["product_id"],
        "search_text": search_text,
        "assigned_shelf_id": product["assigned_shelf_id"],
        "confidence_score": product["confidence_score"],
        "embedding": embedding.tolist()
    })

    
    if idx % 100 == 0:
        print(f"Processed {idx}/{len(products)}")

print("Embeddings generation completed.")

Total products: 1351
Processed 0/1351
Processed 100/1351
Processed 200/1351
Processed 300/1351
Processed 400/1351
Processed 500/1351
Processed 600/1351
Processed 700/1351
Processed 800/1351
Processed 900/1351
Processed 1000/1351
Processed 1100/1351
Processed 1200/1351
Processed 1300/1351
Embeddings generation completed.


In [7]:
# Save Embeddings Matrix

embedding_matrix = np.array([
    item["embedding"]
    for item in embedding_records
])

np.save("embeddings_matrix.npy", embedding_matrix)

print("Embeddings matrix saved successfully.")

Embeddings matrix saved successfully.


In [9]:
import faiss
import numpy as np


# Load Embeddings Matrix


embeddings = np.load("embeddings_matrix.npy")

print("Embeddings shape:", embeddings.shape)

# Get Embedding Dimension


dimension = embeddings.shape[1]

print("Embedding dimension:", dimension)


# Create FAISS Index


index = faiss.IndexFlatIP(dimension)

# لأننا استخدمنا normalize_embeddings=True
# Inner Product = Cosine Similarity


# Add Embeddings to Index


index.add(embeddings)

print("Total vectors in index:", index.ntotal)


# Save FAISS Index


faiss.write_index(index, "products_faiss.index")

print("FAISS index saved successfully.")

Embeddings shape: (1351, 384)
Embedding dimension: 384
Total vectors in index: 1351
FAISS index saved successfully.


In [17]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# Load Embedding Model

model = SentenceTransformer('all-MiniLM-L6-v2')

# Load FAISS Index


index = faiss.read_index("products_faiss.index")

print("FAISS index loaded successfully.")


# Load Product Metadata


products = []

with open("embeddings_ready.jsonl", "r", encoding="utf-8") as f:

    for line in f:
        products.append(json.loads(line))

print("Products loaded:", len(products))


# Semantic Search Function


def semantic_search(query, top_k=5):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search inside FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        product = products[idx]

        results.append({

            "product_id": product.get("product_id", "N/A"),

            "search_text": product.get("search_text", "N/A"),

            "assigned_shelf_id": product.get("assigned_shelf_id", "N/A"),

            "confidence_score": product.get("confidence_score", 0),

            "similarity_score": float(score)
        })

    return results


# Test Search


results = semantic_search("fast charger for iphone")

print("\nTop Search Results:\n")

for r in results:
    print(r)

FAISS index loaded successfully.
Products loaded: 1351

Top Search Results:

{'product_id': 'N/A', 'search_text': 'N/A', 'assigned_shelf_id': 'N/A', 'confidence_score': 0, 'similarity_score': 0.7029430270195007}
{'product_id': 'N/A', 'search_text': 'N/A', 'assigned_shelf_id': 'N/A', 'confidence_score': 0, 'similarity_score': 0.6412999033927917}
{'product_id': 'N/A', 'search_text': 'N/A', 'assigned_shelf_id': 'N/A', 'confidence_score': 0, 'similarity_score': 0.5770971775054932}
{'product_id': 'N/A', 'search_text': 'N/A', 'assigned_shelf_id': 'N/A', 'confidence_score': 0, 'similarity_score': 0.5761557221412659}
{'product_id': 'N/A', 'search_text': 'N/A', 'assigned_shelf_id': 'N/A', 'confidence_score': 0, 'similarity_score': 0.5653302073478699}


In [19]:
print(products[0])

{'id': 'B07JW9H4J1', 'text': 'Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for i Phone 13 12 11 X 8 7 6 5 Computers and Accessories Accessories and Peripherals Cables and Accessories Cables USB Cables USB Cables Electronics Cables High Compatibility Compatible With i Phone 12 11 X/Xs Max/Xr i Phone 8/8 Plus i Phone 7/7 Plus i Phone 6s/6s Plus i Phon', 'shelf': 'A1', 'score': 0.97}


In [21]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# Load Embedding Model


model = SentenceTransformer('all-MiniLM-L6-v2')


# Load FAISS Index


index = faiss.read_index("products_faiss.index")

print("FAISS index loaded successfully.")

# Load Product Metadata

products = []

with open("embeddings_ready.jsonl", "r", encoding="utf-8") as f:

    for line in f:
        products.append(json.loads(line))

print("Products loaded:", len(products))

# Semantic Search Function

def semantic_search(query, top_k=5):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search inside FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        product = products[idx]

        results.append({

            "product_id": product["id"],

            "search_text": product["text"],

            "assigned_shelf_id": product["shelf"],

            "confidence_score": product["score"],

            "similarity_score": float(score)
        })

    return results

# Test Search

results = semantic_search("fast charger for iphone")

print("\nTop Search Results:\n")

for r in results:
    print(r)

FAISS index loaded successfully.
Products loaded: 1351

Top Search Results:

{'product_id': 'B07XLCFSSN', 'search_text': 'Amazonbasics Nylon Braided Usb-C To Lightning Cable Fast Charging Mfi Certified Smartphone Iphone Charger 6-Foot Dar Computers and Accessories Accessories and Peripherals Cables and Accessories Cables USB Cables USB Cables Chargers and Adapters Fast Charge When Used With An 18W Or Higher Usb-C Wall Charger With Power Delivery You Can Charge Your Iphone To 50 Batt', 'assigned_shelf_id': 'A2', 'confidence_score': 0.96, 'similarity_score': 0.7029430270195007}
{'product_id': 'B08VS3YLRK', 'search_text': 'Portronics Adapto 20 Type C 20W Fast PD/Type C Adapter Charger with Fast Charging for i Phone 12/12 Pro/12 Mini/12 Pro Ma Electronics Mobiles and Accessories Mobile Accessories Chargers Wall Chargers Wall Chargers Chargers and Adapters 20W HIGH-SPEED CHARGING The Adapto 20 is a Type-C Fast Charger designed to charge your i Phone up to 59 in just 30 minut', 'assigned_she

In [23]:
results = semantic_search("bluetooth headphones")

for r in results:
    print(r)

{'product_id': 'B09Y5MP7C4', 'search_text': 'Noise Buds Vs104 Bluetooth Truly Wireless in Ear Earbuds with Mic 30-Hours of Playtime Instacharge 13Mm Driver and Hy Electronics Headphones Earbuds and Accessories Headphones In-Ear In-Ear Audio Accessories Up to 30-hour playtime Get set for a day full of music and then some more Instacharge Enjoy 150 minutes of playtime in j', 'assigned_shelf_id': 'A3', 'confidence_score': 0.98, 'similarity_score': 0.6237154603004456}
{'product_id': 'B09ND94ZRG', 'search_text': 'Boult Audio Airbass Propods X TWS Bluetooth Truly Wireless in Ear Earbuds with Mic 32H Playtime Fast Charging Type-C Electronics Headphones Earbuds and Accessories Headphones In-Ear In-Ear Audio Accessories Note If the size of the earbud tips does not match the size of your ear canals or the headset is not worn properly in yo', 'assigned_shelf_id': 'A3', 'confidence_score': 0.98, 'similarity_score': 0.6125409007072449}
{'product_id': 'B09NR6G588', 'search_text': 'Boult Audio Z Char

In [27]:
import json

# Load Store Layout JSON

with open("store_map_dataset.json", "r", encoding="utf-8") as f:
    store_data = json.load(f)


# Extract Shelves

shelves = store_data["shelves"]

# Shelf Lookup Function


def get_shelf_coordinates(shelf_id):

    for shelf in shelves:

        if shelf["id"] == shelf_id:

            return {
                "shelf_id": shelf["id"],
                "name": shelf["name"],
                "x": shelf["x"],
                "y": shelf["y"],
                "center_x": shelf["center_x"],
                "center_y": shelf["center_y"]
            }

    return None


# Test

result = get_shelf_coordinates("A3")

print(result)

{'shelf_id': 'A3', 'name': 'Audio Accessories', 'x': 10.29, 'y': 41.3, 'center_x': 16.76, 'center_y': 46.74}


In [29]:
# Extract Path Nodes

nodes = store_data["path_nodes"]
# Find Closest Node To Shelf

def find_closest_node_to_shelf(shelf_id):

    # Get shelf coordinates
    shelf = get_shelf_coordinates(shelf_id)

    if not shelf:
        return None

    shelf_x = shelf["center_x"]
    shelf_y = shelf["center_y"]

    closest_node = None
    min_distance = float("inf")

    for node in nodes:

        node_x = node["x"]
        node_y = node["y"]

        # Euclidean distance
        distance = (
            (node_x - shelf_x) ** 2 +
            (node_y - shelf_y) ** 2
        ) ** 0.5

        if distance < min_distance:

            min_distance = distance
            closest_node = node

    return {
        "shelf_id": shelf_id,
        "closest_node_id": closest_node["node_id"],
        "node_label": closest_node["label"],
        "distance": round(min_distance, 2),
        "node_x": closest_node["x"],
        "node_y": closest_node["y"]
    }

# Test

result = find_closest_node_to_shelf("A3")

print(result)

{'shelf_id': 'A3', 'closest_node_id': 'N039', 'node_label': 'entry-A3', 'distance': 7.17, 'node_x': 16.76, 'node_y': 53.91}


In [33]:

import json
import faiss
import heapq
import numpy as np

from sentence_transformers import SentenceTransformer

# LOAD MODEL


model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded.")


# LOAD FAISS INDEX

index = faiss.read_index("products_faiss.index")

print("FAISS index loaded.")

# LOAD PRODUCTS

products = []

with open("embeddings_ready.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        products.append(json.loads(line))

print("Products loaded:", len(products))

# LOAD STORE NAVIGATION FILE

with open("store_layout_navigation.json", "r", encoding="utf-8") as f:
    store_data = json.load(f)

print("Store navigation loaded.")

# EXTRACT NODES + EDGES

nodes = store_data["path_nodes"]
edges = store_data["edges"]

print("Nodes:", len(nodes))
print("Edges:", len(edges))

# BUILD GRAPH

graph = {}

for node in nodes:
    graph[node["node_id"]] = []

for edge in edges:

    from_node = edge["from"]
    to_node = edge["to"]
    distance = edge["distance"]

    graph[from_node].append((to_node, distance))
    graph[to_node].append((from_node, distance))

print("Graph built successfully.")


# NODE COORDINATES LOOKUP

node_lookup = {}

for node in nodes:
    node_lookup[node["node_id"]] = {
        "x": node["x"],
        "y": node["y"],
        "label": node["label"]
    }

# SHELF -> NODE MAPPING

shelf_to_node = {
    "A1": "N037",
    "A2": "N038",
    "A3": "N039",
    "A4": "N040",

    "B1": "N041",
    "B2": "N042",
    "B3": "N043",
    "B4": "N044",

    "C1": "N045",
    "C2": "N046",
    "C3": "N047",
    "C4": "N048",

    "D1": "N049",
    "D2": "N050",
    "D3": "N051",
    "D4": "N052"
}

print("Shelf mapping ready.")

# SEMANTIC SEARCH

def search_product(query, top_k=1):

    query_embedding = model.encode([query])

    scores, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    idx = indices[0][0]
    score = scores[0][0]

    product = products[idx]

    return {
        "product_id": product.get("id", "N/A"),
        "text": product.get("text", "N/A"),
        "shelf": product.get("shelf", "N/A"),
        "score": product.get("score", 0),
        "similarity_score": float(score)
    }


# DIJKSTRA ALGORITHM

def shortest_path(graph, start, end):

    queue = [(0, start, [])]

    visited = set()

    while queue:

        cost, node, path = heapq.heappop(queue)

        if node in visited:
            continue

        visited.add(node)

        path = path + [node]

        if node == end:
            return cost, path

        for neighbor, weight in graph[node]:

            if neighbor not in visited:
                heapq.heappush(
                    queue,
                    (cost + weight, neighbor, path)
                )

    return float("inf"), []


# GET ROUTE COORDINATES

def get_route_coordinates(path):

    coordinates = []

    for node_id in path:

        node = node_lookup[node_id]

        coordinates.append({
            "node_id": node_id,
            "x": node["x"],
            "y": node["y"],
            "label": node["label"]
        })

    return coordinates


# FULL NAVIGATION PIPELINE

def navigate_to_product(query):

    # STEP 1
    product = search_product(query)

    shelf_id = product["shelf"]

    print("\nMatched Product:")
    print(product["text"][:150])

    print("\nShelf:", shelf_id)

    # STEP 2
    destination_node = shelf_to_node[shelf_id]

    print("Destination Node:", destination_node)

    # STEP 3
    start_node = "N036"

    total_distance, path = shortest_path(
        graph,
        start_node,
        destination_node
    )

    # STEP 4
    coordinates = get_route_coordinates(path)

    # FINAL RESULT
    result = {
        "query": query,
        "matched_product": product,
        "destination_shelf": shelf_id,
        "destination_node": destination_node,
        "total_distance": total_distance,
        "path": path,
        "coordinates": coordinates
    }

    return result

# TEST

result = navigate_to_product("iphone charger")

print("\n========================")
print("FINAL NAVIGATION RESULT")
print("========================\n")

print("Distance:", result["total_distance"])

print("\nPath:\n")
print(result["path"])

print("\nCoordinates:\n")

for c in result["coordinates"]:
    print(c)

Model loaded.
FAISS index loaded.
Products loaded: 1351
Store navigation loaded.
Nodes: 55
Edges: 112
Graph built successfully.
Shelf mapping ready.

Matched Product:
Amazonbasics Nylon Braided Usb-C To Lightning Cable Fast Charging Mfi Certified Smartphone Iphone Charger 6-Foot Dar Computers and Accessories Accesso

Shelf: A2
Destination Node: N038

FINAL NAVIGATION RESULT

Distance: 82.88400000000001

Path:

['N036', 'N030', 'N027', 'N026', 'N040', 'N039', 'N038']

Coordinates:

{'node_id': 'N036', 'x': 50.0, 'y': 94.13, 'label': 'csh-EN'}
{'node_id': 'N030', 'x': 50.0, 'y': 79.35, 'label': 'bot-EN'}
{'node_id': 'N027', 'x': 45.44, 'y': 79.35, 'label': 'bot-BC'}
{'node_id': 'N026', 'x': 26.32, 'y': 79.35, 'label': 'bot-AB'}
{'node_id': 'N040', 'x': 16.76, 'y': 69.13, 'label': 'approach-A4'}
{'node_id': 'N039', 'x': 16.76, 'y': 53.91, 'label': 'approach-A3'}
{'node_id': 'N038', 'x': 16.76, 'y': 38.7, 'label': 'approach-A2'}


In [35]:

# CONVERT ROUTE TO SVG POINTS

def route_to_svg_points(coordinates):

    points = []

    for point in coordinates:

        # convert store coordinates to SVG pixels
        svg_x = point["x"] * 6.8
        svg_y = point["y"] * 4.6

        points.append(f"{svg_x},{svg_y}")

    return " ".join(points)

# GENERATE SVG ROUTE

svg_route = route_to_svg_points(
    result["coordinates"]
)

print("\nSVG Route:\n")

print(svg_route)


SVG Route:

340.0,432.99799999999993 340.0,365.00999999999993 308.99199999999996,365.00999999999993 178.976,365.00999999999993 113.968,317.99799999999993 113.968,247.98599999999996 113.968,178.02


In [39]:
pip install streamlit svgwrite

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.
